# Day 3 — Delta Lake Advanced (History, Time Travel, MERGE, OPTIMIZE)

**Prerequisites:** Run **`01-Day3-Medallion-Bronze-Silver-Gold`** first so **Silver** and **Gold** Delta tables exist — or re-run the path + Bronze/Silver cells there.

Uses the same **ABFS + `%run ./02-Mount-Azure-Data-Lake`** setup as Day 1 / Day 2.

**Aligned with external labs:** **`09-DeltaTableVersioning`** (history, `timestampAsOf`, delete/restore ideas), **`08-Reading Writing to Delta Tables`** (`MERGE` SQL, `DeltaTable`, `autoOptimize`), **`01-Databricks-datalake-spark`** (lake + SQL patterns). Paths here use **`delta.\`abfss://...\``** instead of `/mnt/...` or S3 URIs.

In [ ]:
%run ./02-Mount-Azure-Data-Lake

In [ ]:
BASE_PATH = base_path
DAY03_ROOT = BASE_PATH + "/day03"
BRONZE_PATH = DAY03_ROOT + "/bronze/flights_raw"
SILVER_PATH = DAY03_ROOT + "/silver/flights_clean"
GOLD_PATH = DAY03_ROOT + "/gold/flights_by_destination"

print("SILVER_PATH:", SILVER_PATH)

## 1 — Table history (`DESCRIBE HISTORY`)

Every Delta write creates a **new version**. Inspect commits for auditing and debugging.

In [ ]:
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, SILVER_PATH)
dt.history(10).select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

## 2 — Time travel reads

Read **logical snapshots** by **version** or **timestamp** (great for reproducing reports or rollback analysis).

In [ ]:
from pyspark.sql.functions import col

ver = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_PATH}`").selectExpr("max(version) as v").collect()[0]["v"]
print("Latest Silver version:", ver)

spark.read.format("delta").option("versionAsOf", int(ver)).load(SILVER_PATH).groupBy("DEST_COUNTRY_NAME").count().orderBy(
    col("count").desc()
).show(5, truncate=False)

### 2b — Time travel by **timestamp** (`timestampAsOf`) — Lab *09* pattern

External versioning lab reads Delta with **`timestampAsOf`** (as well as `versionAsOf`). Below we take the **timestamp of version 0** from history and read that snapshot.

In [ ]:
from pyspark.sql.functions import col as C

hist_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_PATH}`")
ts_v0 = hist_df.filter(C("version") == 0).select("timestamp").collect()
if ts_v0:
    ts = ts_v0[0]["timestamp"]
    ts_lit = str(ts)
    print("Reading Silver TIMESTAMP AS OF:", ts_lit)
    cnt = (
        spark.read.format("delta")
        .option("timestampAsOf", ts_lit)
        .load(SILVER_PATH)
        .count()
    )
    print("Row count at that timestamp:", cnt)
else:
    print("No version 0 in history yet — run notebook 01 first.")

## 3 — `MERGE` (upsert) into Silver

**Pattern:** match on **`route_key`**, update when matched, insert new routes.

This cell **increments `count` by 1** for a few rows to simulate a correction feed, then merges into Silver.

In [ ]:
from pyspark.sql.functions import col, lit

silver = DeltaTable.forPath(spark, SILVER_PATH)

updates = (
    spark.read.format("delta")
    .load(SILVER_PATH)
    .filter(col("DEST_COUNTRY_NAME") == "United States")
    .limit(3)
    .withColumn("count", col("count") + lit(1))
)

(
    silver.alias("t")
    .merge(updates.alias("s"), "t.route_key = s.route_key")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Merge complete. New history:")
DeltaTable.forPath(spark, SILVER_PATH).history(3).select("version", "operation").show(truncate=False)

### 3b — **SQL `MERGE`** on a Delta path (Lab *08* pattern)

The external *Reading Writing Delta* lab uses **`MERGE INTO ... USING ...`**. On Databricks you can target **`delta.\`abfss://...\``**. Example (run only if you register a staging view first):

```sql
-- MERGE INTO delta.`abfss://CONTAINER@ACCOUNT.dfs.core.windows.net/data/day03/silver/flights_clean` AS t
-- USING merge_staging AS s
-- ON t.route_key = s.route_key
-- WHEN MATCHED THEN UPDATE SET *
-- WHEN NOT MATCHED THEN INSERT *
```

You already performed the same logic **in Python** above (`DeltaTable.merge`). Prefer **one style** per pipeline for readability.

### 3c — **Conditional** `whenMatched` (Lab *08* extension)

In Python API you can update only when a condition holds, e.g. avoid overwriting certain rows:

```python
# .whenMatchedUpdate(
#     condition = "s.ingestion_ts > t.ingestion_ts",
#     set = { "count": "s.count", "ingestion_ts": "s.ingestion_ts" }
# )
```

Use this for **slowly changing dimensions** or **late-arriving corrections**.

### 3d — `DELETE` + logical **restore** (Lab *09* ideas)

* **`DELETE`** on Delta creates a new table version (try in SQL: `DELETE FROM delta.\`...\` WHERE ...` with a safe predicate).
* **Restore** an older state by **inserting from time travel** (pattern from lab *09*):

```sql
-- INSERT INTO my_table
-- SELECT * FROM delta.`abfss://.../silver/...` TIMESTAMP AS OF '2025-01-01 12:00:00'
```

Do **not** run destructive SQL in shared training workspaces without instructor approval.

## 4 — Schema evolution (append with new column)

Delta can **evolve schema** when appends introduce new fields (use with care in production).

Below we append to **Bronze** with an extra column `batch_id`.

In [ ]:
from pyspark.sql.functions import coalesce, col, current_timestamp, lit

_src = BASE_PATH + "/2010-summary.csv"
extra = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(_src)
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", coalesce(col("_metadata.file_path"), lit(_src)))
    .withColumn("source_system", lit("day3_schema_evolution_demo"))
    .withColumn("batch_id", lit("batch_002"))
)

extra.write.format("delta").mode("append").option("mergeSchema", "true").save(BRONZE_PATH)

spark.read.format("delta").load(BRONZE_PATH).printSchema()

## 5 — `OPTIMIZE` (Databricks)

**`OPTIMIZE`** compacts small files. Run on Silver after many small writes.

> **Note:** Available on Databricks Delta. On open-source Spark without Databricks runtime, skip this cell or use OSS Delta maintenance APIs.

### 5b — `autoOptimize` / `optimizeWrite` (Lab *08* discussion)

External lab may set session or table properties such as **`delta.autoOptimize.optimizeWrite`** for automatic file sizing on writes. In class, compare **explicit `OPTIMIZE`** vs **auto** trade-offs (cost, predictability).

In [ ]:
spark.sql(f"OPTIMIZE delta.`{SILVER_PATH}`")

## 6 — `VACUUM` (discussion only)

`VACUUM` removes old files no longer referenced by the latest table state. **Default retention is 7 days** — do not run aggressive vacuum in shared training workspaces without instructor approval.

```python
# spark.sql(f"VACUUM delta.`{SILVER_PATH}` RETAIN 168 HOURS")
```

## Summary

| Topic | Why it matters |
|-------|----------------|
| History | Auditability, replay, debugging |
| Time travel | Reproducible reads, rollback analysis |
| MERGE | Idempotent upserts into Silver |
| Schema evolution | Safer iterative ingestion |
| OPTIMIZE | Fewer small files, better read performance |

**End of Day 3 advanced notebook.**